# BNPL Default Prediction — Feature Selection Diagnostic
## Notebook 02: Statistical Evidence Before Data Preparation

**File structure expected:**
```
bnpl_project/
├── data/
│   └── raw/
│       └── accepted_2007_to_2018Q4.csv.gz
├── notebooks/
│   └── 02_feature_selection.ipynb   <-- this file
```

**Rule for this notebook:** Everything here runs ONLY on the
**training period (2013–2015)**. We never let validation or test
data influence feature selection — that would be a subtler form
of the same leakage problem from Notebook 01.

**Output of this notebook:** A ranked feature comparison table
and a finalized feature list to carry into data preparation.


.\bnpl\Scripts\activate

## Section 1: Setup and Load Training Period Only

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import (
    f_classif, chi2, mutual_info_classif
)
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)

print("Libraries loaded")

Libraries loaded


In [2]:
# ─────────────────────────────────────────────────────────────────────
# Load the FULL file but filter to TRAINING YEARS ONLY (2013-2015)
# We read in chunks to avoid loading all 2.2M rows x 151 cols into memory
# at once, then keep only rows where issue_d falls in the training window.
# ─────────────────────────────────────────────────────────────────────

FILE_PATH = '../data/raw/accepted_2007_to_2018Q4.csv.gz'   # adjust if needed

# Columns we need at this stage — candidate features + target + time axis
# (leakage and junk columns already excluded based on Notebook 01 findings)
CANDIDATE_COLS = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
    'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'purpose', 'addr_state', 'dti',
    'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc',
    'issue_d', 'loan_status'
]

chunks = []
chunk_iter = pd.read_csv(FILE_PATH, usecols=CANDIDATE_COLS,
                          chunksize=200_000, low_memory=False)

for chunk in chunk_iter:
    chunk['issue_d_parsed'] = pd.to_datetime(chunk['issue_d'],
                                              format='%b-%Y', errors='coerce')
    chunk['issue_year'] = chunk['issue_d_parsed'].dt.year
    # Keep only training years
    mask = chunk['issue_year'].isin([2013, 2014, 2015])
    if mask.any():
        chunks.append(chunk[mask])

df_train_raw = pd.concat(chunks, ignore_index=True)
print(f"Training period rows loaded: {len(df_train_raw):,}")
print(f"Years present: {sorted(df_train_raw['issue_year'].unique())}")

Training period rows loaded: 791,538
Years present: [np.float64(2013.0), np.float64(2014.0), np.float64(2015.0)]


In [3]:
# Filter to completed loans only, build binary target
df_train_raw = df_train_raw[
    df_train_raw['loan_status'].isin(['Fully Paid', 'Charged Off'])
].copy()

df_train_raw['default'] = (df_train_raw['loan_status'] == 'Charged Off').astype(int)

print(f"Rows after filtering to completed loans: {len(df_train_raw):,}")
print()
print("Class balance in TRAINING period only:")
print(df_train_raw['default'].value_counts(normalize=True).mul(100).round(1))

Rows after filtering to completed loans: 733,451

Class balance in TRAINING period only:
default
0   81.2000
1   18.8000
Name: proportion, dtype: float64


---
## Section 2: Minimal Encoding (Just For Testing — Not Final Prep)

Statistical tests need numeric input. This is NOT your final encoding
strategy — that comes later in data preparation. This is just enough
encoding to run the diagnostic tests.


In [4]:
df_test = df_train_raw.copy()

# emp_length: convert text to numeric scale (simple version for testing)
emp_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
df_test['emp_length_num'] = df_test['emp_length'].map(emp_map)

# grade: ordinal A-G -> 0-6
grade_map = {g: i for i, g in enumerate(['A','B','C','D','E','F','G'])}
df_test['grade_num'] = df_test['grade'].map(grade_map)

# Simple label encoding for nominal categoricals (just for testing)
nominal_cats = ['home_ownership', 'verification_status', 'purpose',
                 'addr_state', 'term', 'sub_grade']
le_dict = {}
for col in nominal_cats:
    le = LabelEncoder()
    df_test[col + '_enc'] = le.fit_transform(df_test[col].astype(str))
    le_dict[col] = le

print("Encoding complete. Columns added:")
print([c for c in df_test.columns if c.endswith(('_num','_enc'))])

Encoding complete. Columns added:
['emp_length_num', 'grade_num', 'home_ownership_enc', 'verification_status_enc', 'purpose_enc', 'addr_state_enc', 'term_enc', 'sub_grade_enc']


---
## Section 3: Univariate Statistical Tests

**Numeric features → ANOVA F-test**
**Categorical features → Chi-Square test**

Each feature tested ALONE against the target, ignoring all other features.


In [5]:
numeric_features = [
    'loan_amnt', 'funded_amnt', 'int_rate', 'installment', 'annual_inc',
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'emp_length_num', 'grade_num'
]
numeric_features = [f for f in numeric_features if f in df_test.columns]

# Drop rows with NaN for this test only (median fill would distort F-test)
df_anova = df_test[numeric_features + ['default']].dropna()

X_num = df_anova[numeric_features]
y = df_anova['default']

f_scores, p_values = f_classif(X_num, y)

anova_results = pd.DataFrame({
    'feature': numeric_features,
    'f_score': f_scores,
    'p_value': p_values
}).sort_values('f_score', ascending=False)

anova_results['significant'] = anova_results['p_value'] < 0.05

print("ANOVA F-test results (numeric features):")
print(anova_results.to_string(index=False))
print()
print(f"Rows used (after dropping NaN for this test): {len(df_anova):,}")

ANOVA F-test results (numeric features):
        feature    f_score  p_value  significant
      grade_num 53310.1247   0.0000         True
       int_rate 49840.9129   0.0000         True
 fico_range_low 10475.2906   0.0000         True
fico_range_high 10475.1397   0.0000         True
            dti  8348.2925   0.0000         True
 inq_last_6mths  3011.4107   0.0000         True
    funded_amnt  2880.5303   0.0000         True
      loan_amnt  2880.4883   0.0000         True
     revol_util  1822.8731   0.0000         True
     annual_inc  1162.4225   0.0000         True
       open_acc   975.3416   0.0000         True
    installment   805.7895   0.0000         True
      revol_bal   328.4358   0.0000         True
        pub_rec   270.5521   0.0000         True
    delinq_2yrs   143.1461   0.0000         True
 emp_length_num    56.9758   0.0000         True
      total_acc    13.0903   0.0003         True

Rows used (after dropping NaN for this test): 693,539


In [6]:
categorical_features = ['home_ownership_enc', 'verification_status_enc',
                        'purpose_enc', 'addr_state_enc', 'term_enc',
                        'sub_grade_enc']
categorical_features = [f for f in categorical_features if f in df_test.columns]

df_chi = df_test[categorical_features + ['default']].dropna()
X_cat = df_chi[categorical_features]
y_cat = df_chi['default']

# Chi2 requires non-negative values — label encoding already gives that
chi_scores, chi_p = chi2(X_cat, y_cat)

chi_results = pd.DataFrame({
    'feature': [f.replace('_enc','') for f in categorical_features],
    'chi2_score': chi_scores,
    'p_value': chi_p
}).sort_values('chi2_score', ascending=False)

chi_results['significant'] = chi_results['p_value'] < 0.05

print("Chi-Square test results (categorical features):")
print(chi_results.to_string(index=False))

Chi-Square test results (categorical features):
            feature  chi2_score  p_value  significant
          sub_grade 206884.0266   0.0000         True
               term  23431.9111   0.0000         True
verification_status   2750.3270   0.0000         True
     home_ownership   1263.0940   0.0000         True
            purpose    564.4309   0.0000         True
         addr_state     53.0564   0.0000         True


---
## Section 4: Mutual Information

Catches NONLINEAR relationships that ANOVA / correlation miss.
Compare this ranking against Section 3 — disagreements are informative.


In [7]:
all_test_features = numeric_features + categorical_features
df_mi = df_test[all_test_features + ['default']].dropna()

X_mi = df_mi[all_test_features]
y_mi = df_mi['default']

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)

mi_results = pd.DataFrame({
    'feature': [f.replace('_enc','').replace('_num','') for f in all_test_features],
    'mutual_info': mi_scores
}).sort_values('mutual_info', ascending=False)

print("Mutual Information scores (all candidate features):")
print(mi_results.to_string(index=False))
print()
print("COMPARE: Any feature ranked LOW in ANOVA/Chi2 but HIGH here?")
print("         That signals a nonlinear relationship worth keeping.")

Mutual Information scores (all candidate features):
            feature  mutual_info
              grade       0.0550
     home_ownership       0.0488
verification_status       0.0447
          sub_grade       0.0428
           int_rate       0.0401
               term       0.0387
        installment       0.0322
            purpose       0.0287
         emp_length       0.0275
     inq_last_6mths       0.0176
    fico_range_high       0.0147
     fico_range_low       0.0140
          loan_amnt       0.0107
        funded_amnt       0.0094
           open_acc       0.0064
                dti       0.0063
         addr_state       0.0062
         annual_inc       0.0039
        delinq_2yrs       0.0035
            pub_rec       0.0032
          total_acc       0.0026
         revol_util       0.0018
          revol_bal       0.0003

COMPARE: Any feature ranked LOW in ANOVA/Chi2 but HIGH here?
         That signals a nonlinear relationship worth keeping.


---
## Section 5: Multicollinearity Check (VIF)

Tells you if a feature is redundant because it's highly correlated
with OTHER FEATURES, not the target. High VIF = redundant information.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_features = [f for f in numeric_features if f != 'grade_num']  # grade_num kept separately
df_vif = df_test[vif_features].dropna()

vif_data = pd.DataFrame()
vif_data['feature'] = vif_features
vif_data['VIF'] = [variance_inflation_factor(df_vif.values, i)
                    for i in range(len(vif_features))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("Variance Inflation Factor (numeric features):")
print(vif_data.to_string(index=False))
print()
print("FLAG: VIF > 10 indicates problematic redundancy")
print("Expected suspects: fico_range_low vs fico_range_high,")
print("                    loan_amnt vs funded_amnt vs installment")

Variance Inflation Factor (numeric features):
        feature           VIF
fico_range_high 33319365.2611
 fico_range_low 33285650.6570
    funded_amnt 11574033.3476
      loan_amnt 11573929.2718
    installment       46.1895
       int_rate       14.0329
       open_acc       12.3957
      total_acc       11.6993
     revol_util       10.1356
            dti        7.4975
 emp_length_num        3.8191
     annual_inc        3.0183
      revol_bal        2.0433
 inq_last_6mths        1.7250
        pub_rec        1.2242
    delinq_2yrs        1.2190

FLAG: VIF > 10 indicates problematic redundancy
Expected suspects: fico_range_low vs fico_range_high,
                    loan_amnt vs funded_amnt vs installment


In [9]:
# Direct correlation check on the suspected redundant pairs
suspect_pairs = [
    ('fico_range_low', 'fico_range_high'),
    ('loan_amnt', 'funded_amnt'),
    ('loan_amnt', 'installment'),
    ('open_acc', 'total_acc'),
]

print("Direct pairwise correlation check:")
for a, b in suspect_pairs:
    if a in df_test.columns and b in df_test.columns:
        corr = df_test[[a, b]].dropna().corr().iloc[0, 1]
        flag = "  << HIGHLY REDUNDANT, consider combining/dropping one" if abs(corr) > 0.9 else ""
        print(f"  {a:<20} vs {b:<20}  corr = {corr:+.3f}{flag}")

Direct pairwise correlation check:
  fico_range_low       vs fico_range_high       corr = +1.000  << HIGHLY REDUNDANT, consider combining/dropping one
  loan_amnt            vs funded_amnt           corr = +1.000  << HIGHLY REDUNDANT, consider combining/dropping one
  loan_amnt            vs installment           corr = +0.952  << HIGHLY REDUNDANT, consider combining/dropping one
  open_acc             vs total_acc             corr = +0.693


---
## Section 6: Variance Threshold

Catches near-constant features automatically — no business judgment needed.


In [10]:
from sklearn.feature_selection import VarianceThreshold

# Normalize scale first so variance is comparable across features
df_var = df_test[numeric_features].dropna()
df_var_normalized = (df_var - df_var.mean()) / df_var.std()

variances = df_var_normalized.var().sort_values()

print("Normalized variance per feature (lowest first):")
print(variances.to_string())
print()
print("FLAG: Variance near 0 means the feature barely changes across")
print("      rows and likely carries little to no separating signal.")

Normalized variance per feature (lowest first):
delinq_2yrs       1.0000
pub_rec           1.0000
open_acc          1.0000
loan_amnt         1.0000
dti               1.0000
fico_range_low    1.0000
annual_inc        1.0000
funded_amnt       1.0000
revol_bal         1.0000
revol_util        1.0000
inq_last_6mths    1.0000
fico_range_high   1.0000
total_acc         1.0000
emp_length_num    1.0000
installment       1.0000
int_rate          1.0000
grade_num         1.0000

FLAG: Variance near 0 means the feature barely changes across
      rows and likely carries little to no separating signal.


---
## Section 7: Baseline Model — Feature Importance & Permutation Importance

This step captures interactions and nonlinear effects that
all previous univariate methods miss entirely.


In [11]:
model_features = numeric_features + categorical_features
df_model = df_test[model_features + ['default']].dropna()

X = df_model[model_features]
y = df_model['default']

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Quick baseline XGBoost — not tuned, just for feature signal
baseline_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
    eval_metric='auc',
    random_state=42
)
baseline_model.fit(X_tr, y_tr)

from sklearn.metrics import roc_auc_score
val_auc = roc_auc_score(y_val, baseline_model.predict_proba(X_val)[:, 1])
print(f"Baseline AUC (diagnostic only, not your final model): {val_auc:.4f}")

Baseline AUC (diagnostic only, not your final model): 0.7319


In [12]:
# Built-in XGBoost feature importance (gain-based)
importance_df = pd.DataFrame({
    'feature': model_features,
    'xgb_importance': baseline_model.feature_importances_
}).sort_values('xgb_importance', ascending=False)

print("XGBoost built-in feature importance:")
print(importance_df.to_string(index=False))

XGBoost built-in feature importance:
                feature  xgb_importance
              grade_num          0.3990
          sub_grade_enc          0.3040
               term_enc          0.1485
     home_ownership_enc          0.0270
         fico_range_low          0.0134
                    dti          0.0127
              loan_amnt          0.0093
             annual_inc          0.0091
         addr_state_enc          0.0084
         inq_last_6mths          0.0080
               open_acc          0.0074
verification_status_enc          0.0071
              revol_bal          0.0067
            installment          0.0067
               int_rate          0.0056
            delinq_2yrs          0.0047
                pub_rec          0.0046
         emp_length_num          0.0046
            purpose_enc          0.0045
              total_acc          0.0044
             revol_util          0.0043
            funded_amnt          0.0000
        fico_range_high          0.0000


In [13]:
# Permutation importance — less biased toward high-cardinality features
perm_result = permutation_importance(
    baseline_model, X_val, y_val,
    n_repeats=10, random_state=42, scoring='roc_auc', n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': model_features,
    'perm_importance_mean': perm_result.importances_mean,
    'perm_importance_std': perm_result.importances_std
}).sort_values('perm_importance_mean', ascending=False)

print("Permutation importance (more robust against cardinality bias):")
print(perm_df.to_string(index=False))
print()
print("COMPARE: Does addr_state rank high in XGBoost importance")
print("         but low here? That would indicate cardinality bias.")

Permutation importance (more robust against cardinality bias):
                feature  perm_importance_mean  perm_importance_std
          sub_grade_enc                0.0623               0.0015
               term_enc                0.0241               0.0008
               int_rate                0.0174               0.0005
              loan_amnt                0.0072               0.0004
                    dti                0.0068               0.0004
         fico_range_low                0.0064               0.0004
     home_ownership_enc                0.0059               0.0003
              revol_bal                0.0053               0.0003
              grade_num                0.0051               0.0004
               open_acc                0.0047               0.0002
             annual_inc                0.0040               0.0003
         addr_state_enc                0.0034               0.0003
              total_acc                0.0017               0.0001

---
## Section 8: Compile Final Comparison Table

This is the evidence document. Every keep/drop decision should
trace back to a row in this table.


In [17]:
import os
os.makedirs('../reports', exist_ok=True)

def clean_name(f):
    return f.replace('_enc', '').replace('_num', '')

# Build unified comparison table
all_features_clean = sorted(set(
    [clean_name(f) for f in numeric_features] +
    [clean_name(f) for f in categorical_features]
))

comparison = pd.DataFrame({'feature': all_features_clean})

# Merge ANOVA results
comparison = comparison.merge(
    anova_results[['feature','f_score','p_value']].rename(
        columns={'f_score':'anova_f','p_value':'anova_p'}),
    on='feature', how='left')

# Merge Chi2 results
chi_results_clean = chi_results.rename(columns={'chi2_score':'chi2','p_value':'chi2_p'})
comparison = comparison.merge(chi_results_clean[['feature','chi2','chi2_p']],
                               on='feature', how='left')

# Merge Mutual Info
mi_clean = mi_results.rename(columns={'mutual_info':'mutual_info'})
comparison = comparison.merge(mi_clean, on='feature', how='left')

# Merge XGB importance
imp_clean = importance_df.copy()
imp_clean['feature'] = imp_clean['feature'].apply(clean_name)
comparison = comparison.merge(
    imp_clean.rename(columns={'xgb_importance':'xgb_importance'}),
    on='feature', how='left')

# Merge permutation importance
perm_clean = perm_df.copy()
perm_clean['feature'] = perm_clean['feature'].apply(clean_name)
comparison = comparison.merge(
    perm_clean[['feature','perm_importance_mean']],
    on='feature', how='left')

# Rank columns (lower rank number = more important), avg across available scores
rank_cols = ['anova_f', 'chi2', 'mutual_info', 'xgb_importance', 'perm_importance_mean']
for col in rank_cols:
    comparison[col + '_rank'] = comparison[col].rank(ascending=False)

comparison['avg_rank'] = comparison[[c+'_rank' for c in rank_cols]].mean(axis=1, skipna=True)
comparison = comparison.sort_values('avg_rank')

pd.set_option('display.max_rows', 30)
print("FINAL FEATURE COMPARISON TABLE (sorted by average rank):")
print(comparison[['feature','avg_rank','anova_f','chi2','mutual_info',
                  'xgb_importance','perm_importance_mean']].to_string(index=False))

# Save this — it's your evidence document
comparison.to_csv('../reports/feature_selection_table.csv', index=False)
print()
print("Saved to: ../reports/feature_selection_table.csv")

FINAL FEATURE COMPARISON TABLE (sorted by average rank):
            feature  avg_rank    anova_f        chi2  mutual_info  xgb_importance  perm_importance_mean
          sub_grade    2.0000        NaN 206884.0266       0.0428          0.3040                0.0623
               term    3.2500        NaN  23431.9111       0.0387          0.1485                0.0241
              grade    3.6667        NaN         NaN       0.0550          0.3990                0.0051
     home_ownership    4.2500        NaN   1263.0940       0.0488          0.0270                0.0059
           int_rate    6.0000 49840.9129         NaN       0.0401          0.0056                0.0174
     fico_range_low    6.2500 10475.2906         NaN       0.0140          0.0134                0.0064
          loan_amnt    7.7500  2880.4883         NaN       0.0107          0.0093                0.0072
                dti    7.7500  8348.2925         NaN       0.0063          0.0127                0.0068
verific

---
## Section 9: Your Final Decisions

For each feature, record keep/drop AND the reason. Use the table above
plus your own business judgment from Notebook 01. Investigate disagreements
between statistical evidence and intuition before deciding.


In [1]:
feature_decisions = {
    "dti":                  ("keep", "Strong across ANOVA, XGB importance, business sense"),
    "fico_range_low":       ("keep", "Strong signal; fico_range_high is a perfect duplicate (corr=1.0), dropped"),
    "fico_range_high":      ("drop", "Perfect duplicate of fico_range_low (corr=1.0), XGB importance=0"),
    "revol_util":           ("keep", "Weak in stats but strong domain reasoning — investigate further in data prep"),
    "annual_inc":           ("keep", "Moderate across most methods, strong business logic"),
    "loan_amnt":            ("keep", "Strong signal; funded_amnt is a perfect duplicate (corr=1.0), dropped"),
    "funded_amnt":          ("drop", "Perfect duplicate of loan_amnt (corr=1.0), XGB importance=0"),
    "installment":          ("drop", "Derived from loan_amnt+term+int_rate (corr=0.95 with loan_amnt) — redundant"),
    "int_rate":             ("keep", "Strong ANOVA F-score, strong permutation importance"),
    "grade":                ("drop", "Redundant with sub_grade — sub_grade is the more granular version"),
    "sub_grade":            ("keep", "Highest chi2 score of all features, strong XGB + permutation importance"),
    "term":                 ("keep", "Strong across chi2, mutual info, XGB importance, permutation importance"),
    "emp_length":           ("keep", "Moderate signal, reasonable business logic, low cost to keep"),
    "home_ownership":       ("keep", "Strong chi2, strong mutual info, reasonable permutation importance"),
    "verification_status":  ("keep", "Strong chi2, strong mutual info"),
    "purpose":              ("keep", "Moderate chi2, strong mutual info — behavioral signal, keep for interpretability"),
    "addr_state":           ("drop", "Weakest across nearly every method — lowest chi2, low mutual info, low importance"),
    "delinq_2yrs":          ("keep", "Lower scores but directly relevant credit history signal, low cost to keep"),
    "inq_last_6mths":       ("keep", "Strong ANOVA F-score, decent mutual info and importance"),
    "open_acc":             ("keep", "Decent ANOVA, moderate VIF (12.4) — monitor but keep for now"),
    "total_acc":            ("drop", "Weak across all methods, correlated with open_acc (0.69), low marginal value"),
    "pub_rec":              ("keep", "Lower scores but directly relevant to credit risk, low cost to keep"),
    "revol_bal":            ("keep", "Moderate ANOVA, decent permutation importance"),
}

final_keep = [f for f, (d, r) in feature_decisions.items() if d == "keep"]
final_drop = [f for f, (d, r) in feature_decisions.items() if d == "drop"]

print(f"KEEP ({len(final_keep)}): {final_keep}")
print(f"DROP ({len(final_drop)}): {final_drop}")

KEEP (17): ['dti', 'fico_range_low', 'revol_util', 'annual_inc', 'loan_amnt', 'int_rate', 'sub_grade', 'term', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal']
DROP (6): ['fico_range_high', 'funded_amnt', 'installment', 'grade', 'addr_state', 'total_acc']


---
## Summary

| Section | Method | What It Catches |
|---------|--------|-----------------|
| 3 | ANOVA / Chi-Square | Linear/categorical association with target |
| 4 | Mutual Information | Nonlinear relationships |
| 5 | VIF + correlation | Redundancy between features |
| 6 | Variance Threshold | Near-constant, low-signal features |
| 7 | XGBoost + Permutation Importance | Real-world signal including interactions |
| 8 | Combined table | Single evidence document for decisions |

**Reminder:** This entire notebook ran on TRAINING years only (2013–2015).
The finalized feature list now carries forward into data preparation,
where the same train-only-fitting principle applies to imputation,
encoding, and scaling.
